# 3.3 ERA5-Land + Landsat → Base de Datos

Descarga desde Google Earth Engine los datos climáticos (ERA5-Land) y de teledetección (Landsat)
para todas las parcelas de todos los inventarios (IFN2, IFN3, IFN4) y los carga en la tabla
`parcela_inventario_estacion`.

**Datos descargados por parcela × año × estación:**
- **ERA5-Land**: precipitación (sum/mean/min/max/std) + 18 bandas de temperatura (mean/min/max/std)
- **Landsat**: índices NDVI, EVI, NDII, GNDVI (min/max/median/mean/std) + terreno (ELE, SLO, ASP)

**Estrategia**: se procesa año a año (para minimizar el uso de memoria GEE) y se guarda
un CSV intermedio por año en `datos_gee/`. Si el CSV ya existe, se omite la descarga de ese año.

In [ ]:
import os
import sys
import pandas as pd
from sqlalchemy import text
from db import engine, DB_NAME

# Añadir módulos locales al path
sys.path.insert(0, os.path.abspath('.'))

from modules.gee_finder_and_downloader.gee_era5_land import GeeDownloadEra5Data
from modules.gee_finder_and_downloader.gee_landsat import GeeDownloadLandsatData

OUTPUT_DIR = 'datos_gee'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'✓ Módulos cargados')
print(f'DB: {DB_NAME}')
print(f'Directorio de salida: {os.path.abspath(OUTPUT_DIR)}')

In [ ]:
# ── Cargar parcelas con coordenadas + años por inventario ────────────────────
coords = pd.read_sql(
    'SELECT parcela_id, latitud AS latitude, longitud AS longitude FROM parcelas',
    engine
)
pi = pd.read_sql(
    'SELECT parcela_id, inventario_id, ano FROM parcela_inventario',
    engine
)

df = pi.merge(coords, on='parcela_id', how='left')
df['ano'] = pd.to_numeric(df['ano'], errors='coerce').astype('Int64')
df = df.dropna(subset=['latitude', 'longitude', 'ano'])
df['ano'] = df['ano'].astype(int)

print(f'Total filas: {len(df)}')
print(df.groupby('inventario_id')['ano'].agg(['min', 'max', 'count']))

In [ ]:
# ── Descarga año a año (ERA5 + Landsat) ──────────────────────────────────────
# Se guarda un CSV intermedio por año. Si ya existe, se omite.

years_all = sorted(df['ano'].unique())
print(f'Años a procesar: {len(years_all)}  ({min(years_all)}-{max(years_all)})')

failed_years = []

for year in years_all:
    out_csv = os.path.join(OUTPUT_DIR, f'gee_{year}.csv')
    if os.path.exists(out_csv):
        print(f'  ✓ {year}: ya existe {out_csv}')
        continue

    parcelas_year = df[df['ano'] == year][['parcela_id', 'latitude', 'longitude']].copy()
    print(f'\n--- Año {year}: {len(parcelas_year)} parcelas ---')

    try:
        era5 = GeeDownloadEra5Data(parcelas_year)
        df_era5 = era5.get_data(int(year), verbose=True, return_type='dataframe')
        df_era5['inventario_id'] = df.loc[df['ano'] == year, 'inventario_id'].iloc[0]

        landsat = GeeDownloadLandsatData(parcelas_year)
        df_landsat = landsat.download_data(int(year), verbose=True, return_type='dataframe')
        df_landsat['inventario_id'] = df.loc[df['ano'] == year, 'inventario_id'].iloc[0]

        merge_keys = ['parcela_id', 'ano', 'estacion_id', 'inventario_id']
        extra_landsat = [c for c in df_landsat.columns if c not in df_era5.columns]
        df_year = pd.merge(
            df_era5,
            df_landsat[['parcela_id', 'ano', 'estacion_id', 'inventario_id'] + extra_landsat],
            how='inner',
            on=merge_keys
        )
        df_year.to_csv(out_csv, index=False)
        print(f'  ✓ Guardado: {out_csv}  ({len(df_year)} filas)')

    except Exception as e:
        print(f'  ✗ Error en año {year}: {e}')
        failed_years.append(year)

if failed_years:
    print(f'\n⚠ Años con error: {failed_years}')
else:
    print('\n✓ Todos los años procesados correctamente')

In [ ]:
# ── Combinar todos los CSVs intermedios ──────────────────────────────────────
csv_files = sorted([
    os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
    if f.startswith('gee_') and f.endswith('.csv')
])
print(f'CSVs encontrados: {len(csv_files)}')

df_final = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f'Total filas combinadas: {len(df_final)}')
print(f'Columnas: {len(df_final.columns)}')
print(df_final.groupby(['inventario_id', 'estacion_id']).size().rename('n'))

In [ ]:
# ── Cargar en parcela_inventario_estacion ─────────────────────────────────────
# La tabla ya existe (creada por NB3.0). Usamos tabla temporal + INSERT ... ON DUPLICATE KEY UPDATE.

TEMP_TABLE = '_tmp_gee'

with engine.begin() as conn:
    conn.execute(text(f'DROP TABLE IF EXISTS `{TEMP_TABLE}`'))

df_final.to_sql(TEMP_TABLE, engine, if_exists='replace', index=False)

# Columnas de datos (excluir las de identificación)
id_cols = {'parcela_id', 'inventario_id', 'estacion_id', 'ano', 'latitude', 'longitude',
           'start_date', 'end_date'}
data_cols = [c for c in df_final.columns if c not in id_cols]

set_clause = ',\n  '.join([f'`{c}` = t.`{c}`' for c in data_cols])

sql = f"""
    INSERT INTO parcela_inventario_estacion
        ({', '.join([f'`{c}`' for c in df_final.columns])})
    SELECT {', '.join([f't.`{c}`' for c in df_final.columns])}
    FROM `{TEMP_TABLE}` t
    ON DUPLICATE KEY UPDATE
      {set_clause}
"""

with engine.begin() as conn:
    result = conn.execute(text(sql))
    conn.execute(text(f'DROP TABLE IF EXISTS `{TEMP_TABLE}`'))

print(f'✓ Cargadas {len(df_final)} filas en parcela_inventario_estacion')

In [ ]:
# ── Verificación ──────────────────────────────────────────────────────────────
resumen = pd.read_sql("""
    SELECT inventario_id, estacion_id, COUNT(*) AS n
    FROM parcela_inventario_estacion
    GROUP BY inventario_id, estacion_id
    ORDER BY inventario_id, estacion_id
""", engine)

print('Distribución en parcela_inventario_estacion:')
print(resumen.to_string(index=False))